## Define Input Parameters

In [ ]:
from pymongo import MongoClient


city = "Torino"

# Path to the GTFS ZIP file
gtfs_zip_path = "GTFS/gtfs_Torino.zip"

# MongoDB connection
mongo_client = MongoClient("mongodb://localhost:27017/")  # Replace with your MongoDB connection string
db = mongo_client["TransitDatabaseTorino15_Test"]

# Define the date; if None it finds the busiest network day
specific_date = None

# Define the hexagonal grid edge length in kilometers
gridEdge = 0.25  # in kilometers

# OSRM endpoint
OSRM_BASE_URL = "http://localhost:5000"

#Maximum Walking time from point to stop
MAX_WALKING_TIME = 15 * 60

# Walking speed (in meters per second)
WALKING_SPEED = 1.4  # Average human walking speed (5 km/h)
MAX_WALKING_DISTANCE = WALKING_SPEED *  MAX_WALKING_TIME #Max walking distance
aoi_path = r"Boundry\Torino.shp"  # Update with your Area of Interest shapefile path
aoi_POP = r"POP\Torino.shp" # Update with your population shapefile path (Population should be in grid format)
aoi_POI = r"POI\POI_Torino.shp" # Update with your point of Interest shapefile path

MAX_TRAVEL_TIME = 60 * 60 # Maximum travle time
specific_start_hours = [3, 8, 22] #start time of the trip from the point e.g. 8 is 8AM and 22 is 10PM
MAX_WALKING_TIME_STOPS = 5 * 60 #Maximum Walking time from stop to stop (Transfer)
MAX_WALKING_DISTANCE_STOP = WALKING_SPEED * MAX_WALKING_TIME_STOPS #Maximum Walking distance from stop to stop (Transfer)

# Load the data from GTFS Zip File


In [ ]:
from Library.Library import load_gtfs_data

stops, trips, routes, calendar_dates, calendar = load_gtfs_data(gtfs_zip_path)


# Store GTFS on Database

In [ ]:
from Library.Library import process_and_save_gtfs_to_mongo

# Process and save the loaded GTFS data into MongoDB
process_and_save_gtfs_to_mongo(db, calendar_dates, routes, trips, stops, calendar,aoi_path)

## Extract and preprocess GTFS data to build stop-to-stop edge list for a specific date and store in MongoDB


In [ ]:
from Library.Library import process_gtfs_data

#  Run the entire edge creation and saving process
process_gtfs_data(db, gtfs_zip_path, specific_date)

# Create Hexagonal Grid

In [ ]:
from Library.Library import process_hexagonal_grid

process_hexagonal_grid(db, aoi_path, gridEdge)

# Store Population on Database

In [ ]:
from Library.Library import process_grid_population
population_column = "TOT_P_2018" # the attribute showing the population in the population shapefile
# Run the function
process_grid_population(db, aoi_POP, population_column)

## Add population to the hexagons by computing intersection with population grids using $geoIntersects and Shapely


In [ ]:
from Library.Library import process_population_computation

process_population_computation(db)

## Walking time calculation from Points to closest stops within walking distance

In [ ]:
from Library.Library import process_points

process_points(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)

## Walking time calculation from Stops to closest stops and points with walking distance

In [ ]:
from Library.Library import process_stops

process_stops(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE_STOP, MAX_WALKING_TIME_STOPS)

## Create Stop ID Index

In [ ]:
from Library.Library import create_stop_ids
create_stop_ids(db, city, output_dir="matrices")

## Store POI from OSM (You have to either run this or the one below)

In [ ]:
from Library.Library import fetch_pois_in_boundary, store_pois_to_mongodb
# Define groups with their respective OSM tag filters.

groups = {
    "Education": [
        {"key": "amenity", "value": "school"},
        {"key": "amenity", "value": "university"},
        {"key": "amenity", "value": "college"}
    ]
}


pois_gdf = fetch_pois_in_boundary(aoi_path, groups)
store_pois_to_mongodb(pois_gdf, db)

## Store Shapefile of POI locally

In [ ]:
from Library.Library  import store_poi_geometry_to_mongodb

# Call the processing function
store_poi_geometry_to_mongodb(db, aoi_POI)

## Find closest stops to each POI

In [ ]:
from Library.Library  import process_poi_reachable_stops

process_poi_reachable_stops(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)


## Find Closest Point to each POI

In [ ]:
from Library.Library import process_poi_reachable_points, remove_unreachable_pois

process_poi_reachable_points(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)
remove_unreachable_pois(db)

## Compute Accessibility P2POI(Time) | P2POI(#POI) | POI2P(Time) | P2P(Time)

In [ ]:
from Library.Library import process_accessibility_time_to_pois
process_accessibility_time_to_pois(db, city, specific_start_hours,MAX_TRAVEL_TIME)

from Library.Library import process_reachable_poi_counts
process_reachable_poi_counts(db, city, specific_start_hours,MAX_TRAVEL_TIME)

from Library.Library_POI import process_accessibility_from_pois
process_accessibility_from_pois(db, city, specific_start_hours,MAX_TRAVEL_TIME)


from Library.Library_P2P import process_accessibility_average
process_accessibility_average(db, city, specific_start_hours,MAX_TRAVEL_TIME)


## Create Accessibility Map

In [ ]:
thresholds = [30, 45, 60, 90] #Time threshold in minutes
thresholds_POI = [100, 200, 300, 400] # POI number threshold

from Library.Library import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_P2POI.html",thresholds=thresholds,layer_opacity=0.7)

from Library.Library import visualize_multi_hour_accessibility_Num_POI
visualize_multi_hour_accessibility_Num_POI(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_P2NumPOI.html",thresholds=thresholds_POI,layer_opacity=0.7)

from Library.Library_POI import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_POI2P.html",thresholds=thresholds,layer_opacity=0.7)

from Library.Library_P2P import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_P2P.html",thresholds=thresholds,layer_opacity=0.7)


# Equity Analysis (Lorenz & Gini)

Accessibility tells us *where* transit performs; equity asks how fairly that
performance is distributed over **people**. Following the methodology of
[hEART 2022](https://arxiv.org/abs/2206.09037) and
[TRB 2023](https://arxiv.org/abs/2210.00128), we weight each hex cell's
accessibility by its population and compute the Lorenz curve and Gini index.

In [ ]:
import numpy as np
from Library.equity import equity_summary, gini, lorenz_points

# Pull per-hex accessibility and population from the results collection.
hexes = list(db['hex_grid'].find({}, {'accessibility_P2P': 1, 'population': 1}))
acc = np.array([h.get('accessibility_P2P', 0.0) for h in hexes])
pop = np.array([h.get('population', 0.0) for h in hexes])
mask = pop > 0
summary = equity_summary(acc[mask], pop[mask])
summary

In [ ]:
import matplotlib.pyplot as plt

pts = lorenz_points(acc[mask], pop[mask])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], '--', color='grey', label='perfect equality')
ax.plot(pts[:, 0], pts[:, 1], color='#8a6d3b',
        label=f"Torino (Gini = {summary['gini']:.3f})")
ax.set_xlabel('cumulative population share')
ax.set_ylabel('cumulative accessibility share')
ax.legend(); ax.set_title('Transit accessibility Lorenz curve')
plt.tight_layout()